<a href="https://colab.research.google.com/github/yashita-mishra/IML-Assignment/blob/main/A1Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ChatGPT and Gemini were used to use numpy appropriately and to debug the code.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes

In [ ]:
class Node:
  def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value
  def is_leaf_node(self):
    return self.value is not None

In [ ]:
class DecisionTree:
  def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.n_features = n_features
    self.root = None
  def split(self, X_column, thresh):
    #the code below stores the values of features according to their position after splitting
    left_idxs = [i for i, val in enumerate(X_column) if val <= thresh]
    right_idxs = [i for i, val in enumerate(X_column) if val > thresh]
    return left_idxs, right_idxs
  def mse(self, left, right):
    mean_left = sum(left) / len(left)
    mean_right = sum(right) / len(right)
    left_error = sum((x - mean_left)**2 for x in left)
    right_error = sum((x - mean_right)**2 for x in right)
    return left_error + right_error
  def best_split(self, X, y, feat_idxs):
    #returns the feature and threshold that minimises the error
    best_feat = None
    best_thresh = None
    best_error = float('inf') #initialises with a large error
    for feat in feat_idxs:
      values = [row[feat] for row in X]
      for thresh in values:
        left = [y[i] for i in range(len(X)) if X[i][feat] <= thresh]
        right = [y[i] for i in range(len(X)) if X[i][feat] > thresh]
        if len(left) == 0 or len(right) == 0:
          continue
        error = self.mse(left, right)
        if error < best_error:
          best_error = error
          best_feat = feat
          best_thresh = thresh
    return best_feat, best_thresh
  def grow_tree(self, X, y, depth):
    n_samples = len(X)
    n_features = len(X[0])
    if depth >= self.max_depth or n_samples < self.min_samples_split:
      leaf_value = sum(y) / len(y)
      return Node(value=leaf_value)
    feat_idxs = np.random.choice(n_features, self.n_features, replace=False) #randomly selects a subset of features
    #finds the best split using the above features
    best_feat, best_thresh = self.best_split(X, y, feat_idxs)
    #creates a leaf node if no such split exists
    if best_feat is None:
      return Node(value=sum(y) / len(y))
    feature_column = [row[best_feat] for row in X]
    left_idxs, right_idxs = self.split(feature_column, best_thresh)
    left_X = [X[i] for i in left_idxs]
    left_y = [y[i] for i in left_idxs]
    right_X = [X[i] for i in right_idxs]
    right_y = [y[i] for i in right_idxs]
    #Recursively builds left and right subtrees
    left = self.grow_tree(left_X, left_y, depth + 1)
    right = self.grow_tree(right_X, right_y, depth + 1)
    return Node(best_feat, best_thresh, left, right)
  def build_tree(self, X, y):
    X = X.tolist()
    y = y.tolist()
    if self.n_features is None:
      self.n_features = int(np.sqrt(len(X[0])))
    self.root = self.grow_tree(X, y, 0)
  def traversetree(self, X, node):
    if node.is_leaf_node():
      return node.value
    if X[node.feature] <= node.threshold:
      return self.traversetree(X, node.left)
    return self.traversetree(X, node.right)
  def predict(self, X):
    return self.traversetree(X, self.root)

In [ ]:
class RandomForest:
  def __init__(self,n_trees=100, max_depth=10, min_samples_split=2, n_features=None):
    self.n_trees = n_trees
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.n_features = n_features
    self.trees = []
  def build_forest(self, X, y):
    self.trees = []
    X = X.tolist()
    y = y.tolist()
    for i in range(self.n_trees):
      idxs = np.random.choice(len(X), len(X), replace=True)
      X_s = [X[i] for i in idxs]
      Y_s = [y[i] for i in idxs]
      tree = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split, n_features=self.n_features)
      tree.build_tree(np.array(X_s), np.array(Y_s))
      self.trees.append(tree)
  def predict(self, X):
    X = X.tolist()
    preds = []
    for x in X:
      tree_preds = [tree.predict(x) for tree in self.trees]
      preds.append(sum(tree_preds)/len(tree_preds))
    return preds

In [ ]:
data = load_diabetes()
X = data.data
y = data.target

In [ ]:
tree = DecisionTree(max_depth=10)
tree.build_tree(X, y)
rf = RandomForest(n_trees = 100)
rf.build_forest(X, y)

KeyboardInterrupt: 

In [ ]:
def _mse(y_true, y_pred):
    n = len(y_true)
    error = 0
    for i in range(n):
        error += (y_true[i] - y_pred[i]) ** 2
    return error / n

In [ ]:
tree_preds = [tree.predict(x.tolist()) for x in X]
rf_preds = rf.predict(X)
print(tree_preds[:5])
print(rf_preds[:5])

[184.86363636363637, 100.8529411764706, 184.86363636363637, 206.0, 144.5]
[192.33011344927655, 82.06326522633448, 151.2928921913678, 194.50895676304373, 111.5277275538149]


In [ ]:
tree_mse = _mse(y.tolist(), tree_preds)
rf_mse = _mse(y.tolist(), rf_preds)
print("Decision Tree MSE:", tree_mse)
print("Random Forest MSE:", rf_mse)

Decision Tree MSE: 430.49209880373365
Random Forest MSE: 631.9862232133996
